In [1]:
file_name = 'recording_20251029_140939.mp4'
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
file_path = '/content/drive/MyDrive/srl_data/' + file_name

In [3]:
!pip install transformers torch huggingface_hub

In [4]:
# from huggingface_hub import notebook_login

# notebook_login()


In [5]:
import transformers
import torch

In [6]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
model_name = "nvidia/Cosmos-Reason2-2B"
model = transformers.Qwen3VLForConditionalGeneration.from_pretrained(
    model_name, dtype=torch.float16, device_map="auto", attn_implementation="sdpa"
)
processor: transformers.Qwen3VLProcessor = (
    transformers.AutoProcessor.from_pretrained(model_name)
)



#

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/626 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [14]:
video_messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}],
    },
    {"role": "user", "content": [
            {
                "type": "video",
                "video": "/content/drive/MyDrive/srl_data/recording_20251029_140939.mp4",
                "fps": 4,
            },
            {"type": "text", "text": (
                    "Return the normalized [x, y] coordinates for the center of mass of the satellite in the image. The top-left is (0,0) and the bottom-right is (1000,1000). And return the orientation of the satellite [w1, w2, w3] in Euler rotation angles. Also return the normalized velocities in the horizontal and vertical axis as [Vx, Vy], with negative values implying leftward or downward velocities in the axis respectively. Put the answer in a json of the values x, y, w1, w2, w3, Vx, Vy. Answer the question using the following format:\n\n<think>\nYour reasoning.\n</think>\n\nWrite your final answer immediately after the </think> tag."
                )
            },
        ]
    },
]

In [15]:
#  Process inputs
inputs = processor.apply_chat_template(
    video_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    fps=4,
)
inputs = inputs.to(model.device)

# Run inference
generated_ids = model.generate(**inputs, max_new_tokens=4096)
generated_ids_trimmed = [
    out_ids[len(in_ids) :]
    for in_ids, out_ids in zip(inputs.input_ids, generated_ids, strict=False)
]
output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)


In [16]:
output_text

['<think>\nOkay, let\'s break this down. The user is asking for the center of mass of the satellite in the video and its orientation in Euler rotation angles, along with the normalized velocity in the horizontal and vertical axes. \n\nFirst, I need to recall the details provided about the satellite. The video shows that the satellite has a central body with two large rectangular panels on either side, possibly solar panels. The center is slightly elevated, and there\'s a smaller component on top. The structure is symmetrical, with a central body and two panels extending outward.\n\nThe center of mass would be the point around which the satellite is balanced. Since the satellite in the video is symmetrical, the center of mass is likely at the geometric center of the central body. The video shows the center is slightly elevated, so the center of mass might be a bit above the main body. However, given the symmetry, the exact position might be at the center of the central body.\n\nNext, th